# Silver Layer — Admissions

## Source
`cdr_mimiciv.bronze.admissions`

## Target
`cdr_mimiciv.silver.admissions`

The target table is published with CREATE OR REPLACE semantics; downstream Gold depends on it.

## Grain
One row represents one unique hospital admission identified by `hadm_id`.

## Silver objective
Create a reusable and clinically interpretable hospital encounter entity while
preserving source semantics and identifying questionable temporal relationships
without silently removing records.

## Engineering principles
- Preserve source clinical meaning.
- Preserve valid source nulls.
- Do not automatically remove unusual EHR records.
- Standardise datatypes and text representation.
- Validate encounter identifiers and temporal relationships.
- Use flags for questionable records rather than guessing corrections.
- Apply only deterministic reusable transformations.
- Defer cohort definitions and analytical classifications to Gold.

In [0]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TARGET = f"{SILVER}.admissions"

In [0]:
admissions_bronze = spark.table(f"{BRONZE}.admissions")

display(admissions_bronze.limit(20))

In [0]:
# establish the grain
display(
    admissions_bronze.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("hadm_id").alias("unique_hadm_ids"),
        F.countDistinct("subject_id").alias("unique_subject_ids"),
        F.sum(
            F.when(F.col("hadm_id").isNull(), 1).otherwise(0)
        ).alias("null_hadm_ids"),
        F.sum(
            F.when(F.col("subject_id").isNull(), 1).otherwise(0)
        ).alias("null_subject_ids")
    )
)

In [0]:
# explicitly check duplicate admissions
duplicate_admissions = (
    admissions_bronze
    .groupBy("hadm_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate hadm_ids:", duplicate_admissions.count())

In [0]:
# profile missingness
clinical_columns = [
    "subject_id",
    "hadm_id",
    "admittime",
    "dischtime",
    "deathtime",
    "admission_type",
    "admit_provider_id",
    "admission_location",
    "discharge_location",
    "insurance",
    "language",
    "marital_status",
    "race",
    "edregtime",
    "edouttime",
    "hospital_expire_flag"
]

display(
    admissions_bronze.select(
        *[
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in clinical_columns
        ]
    )
)

In [0]:
# inspect categorical values
display(
    admissions_bronze
    .groupBy("admission_type")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
# now test temporal quality
temporal_profile = admissions_bronze.select(

    F.sum(
        F.when(
            F.col("admittime") > F.col("dischtime"),
            1
        ).otherwise(0)
    ).alias("admit_after_discharge"),

    F.sum(
        F.when(
            F.col("edregtime").isNotNull() &
            F.col("edouttime").isNotNull() &
            (F.col("edregtime") > F.col("edouttime")),
            1
        ).otherwise(0)
    ).alias("edreg_after_edout"),

    F.sum(
        F.when(
            F.col("deathtime").isNotNull() &
            (F.col("deathtime") < F.col("admittime")),
            1
        ).otherwise(0)
    ).alias("death_before_admission"),

    F.sum(
        F.when(
            F.col("deathtime").isNotNull() &
            (F.col("deathtime") > F.col("dischtime")),
            1
        ).otherwise(0)
    ).alias("death_after_discharge"),

    F.sum(
        F.when(
            (F.col("hospital_expire_flag") == 1) &
            F.col("deathtime").isNull(),
            1
        ).otherwise(0)
    ).alias("expired_but_no_deathtime")

)

display(temporal_profile)

## Temporal Data Quality Findings

Temporal profiling of the Bronze `admissions` table identified a small number of
records with unusual timestamp relationships:

| Check | Records | Silver treatment |
|---|---:|---|
| Admission time occurs after discharge time | 175 | Preserve the record and flag it |
| ED registration time occurs after ED discharge time | 3 | Preserve the record and flag it |
| Recorded death time occurs before admission time | 26 | Preserve the record and flag it |
| Recorded death time occurs after discharge time | 745 | Preserve the record and flag it |
| `hospital_expire_flag = 1` but `deathtime` is missing | 11 | Preserve the record and flag the inconsistency (the flag also covers `hospital_expire_flag = 0` with a `deathtime`; that component was not profiled) |

### Engineering decision

These records will **not be deleted or automatically corrected in the Silver layer**.

The official documentation notes that organ donor accounts are sometimes created for
patients who died in hospital: distinct admissions with very short, sometimes negative
lengths of stay, whose `deathtime` is frequently the same as the earlier admission's. It also
notes that `deathtime` is almost always the same as `dischtime`, with some discrepancies due
to typographical errors. Whether the flagged records above are such cases has **not** been
profiled.

MIMIC-IV represents real-world EHR data and can contain unusual timestamp
relationships caused by clinical workflows, administrative processes,
documentation timing, de-identification, or source-system inconsistencies.

The purpose of Silver is to make these issues visible without changing the
clinical meaning of the source data. Therefore, the original timestamps are
preserved and explicit data-quality flags are added.

This follows a non-destructive data-quality approach:

`preserve source value → identify anomaly → add DQ flag → allow downstream users to decide`

No timestamp will be guessed, replaced, swapped, or imputed.

### Silver quality fields

The following reusable quality indicators will be created:

- `dq_invalid_admission_time` — discharge occurs before admission.
- `dq_invalid_ed_time` — ED discharge occurs before ED registration.
- `dq_death_before_admission` — recorded death occurs before hospital admission.
- `dq_death_after_discharge` — recorded death occurs after hospital discharge.
- `dq_expire_deathtime_mismatch` — the hospital mortality flag and recorded
  death time are inconsistent.

These flags indicate records that require caution; they do **not automatically
mean that the source record is clinically invalid**.

Project-specific decisions about excluding anomalous records from research
cohorts will be deferred to the Gold or analytical layer.

In [0]:
##Adding metadata to the data quality flag columns and newly created columns in silver.

%sql
ALTER TABLE cdr_mimiciv.silver.admissions
ALTER COLUMN dq_invalid_admission_time
COMMENT 'Data quality flag indicating that the recorded discharge time occurs before the admission time. This flag requires caution but does not automatically mean the source record is clinically invalid.';

ALTER TABLE cdr_mimiciv.silver.admissions
ALTER COLUMN dq_invalid_ed_time
COMMENT 'Data quality flag indicating that the recorded emergency department discharge time occurs before the emergency department registration time. This flag requires caution but does not automatically mean the source record is clinically invalid.';

ALTER TABLE cdr_mimiciv.silver.admissions
ALTER COLUMN dq_death_before_admission
COMMENT 'Data quality flag indicating that the recorded death time occurs before the hospital admission time. This flag requires caution but does not automatically mean the source record is clinically invalid.';

ALTER TABLE cdr_mimiciv.silver.admissions
ALTER COLUMN dq_death_after_discharge
COMMENT 'Data quality flag indicating that the recorded death time occurs after the hospital discharge time. This flag requires caution but does not automatically mean the source record is clinically invalid.';

ALTER TABLE cdr_mimiciv.silver.admissions
ALTER COLUMN dq_expire_deathtime_mismatch
COMMENT 'Data quality flag indicating an inconsistency between the hospital mortality flag and the recorded death time. This flag requires caution but does not automatically mean the source record is clinically invalid.';

In [0]:
admissions_silver = (
    admissions_bronze

    # Keep only the actual admissions fields
    .select(
        "subject_id",
        "hadm_id",
        "admittime",
        "dischtime",
        "deathtime",
        "admission_type",
        "admit_provider_id",
        "admission_location",
        "discharge_location",
        "insurance",
        "language",
        "marital_status",
        "race",
        "edregtime",
        "edouttime",
        "hospital_expire_flag"
    )

    # Admission after discharge
    .withColumn(
        "dq_invalid_admission_time",
        F.when(
            F.col("admittime").isNotNull()
            & F.col("dischtime").isNotNull()
            & (F.col("admittime") > F.col("dischtime")),
            True
        ).otherwise(False)
    )

    # ED registration after ED discharge
    .withColumn(
        "dq_invalid_ed_time",
        F.when(
            F.col("edregtime").isNotNull()
            & F.col("edouttime").isNotNull()
            & (F.col("edregtime") > F.col("edouttime")),
            True
        ).otherwise(False)
    )

    # Death before admission
    .withColumn(
        "dq_death_before_admission",
        F.when(
            F.col("deathtime").isNotNull()
            & F.col("admittime").isNotNull()
            & (F.col("deathtime") < F.col("admittime")),
            True
        ).otherwise(False)
    )

    # Death after discharge
    .withColumn(
        "dq_death_after_discharge",
        F.when(
            F.col("deathtime").isNotNull()
            & F.col("dischtime").isNotNull()
            & (F.col("deathtime") > F.col("dischtime")),
            True
        ).otherwise(False)
    )

    # Mortality flag / deathtime disagreement
    .withColumn(
        "dq_expire_deathtime_mismatch",
        F.when(
            (F.col("hospital_expire_flag") == 1)
            & F.col("deathtime").isNull(),
            True
        )
        .when(
            (F.col("hospital_expire_flag") == 0)
            & F.col("deathtime").isNotNull(),
            True
        )
        .otherwise(False)
    )

    # Genuine Silver processing metadata
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

In [0]:
# Flag counts on the Silver DataFrame (single pass)
display(admissions_silver.limit(20))

PROFILED_FLAG_COUNTS = {
    "dq_invalid_admission_time": 175,
    "dq_invalid_ed_time": 3,
    "dq_death_before_admission": 26,
    "dq_death_after_discharge": 745,
}
PROFILED_EXPIRED_NO_DEATHTIME = 11  # hospital_expire_flag = 1 and deathtime IS NULL

FLAG_COLUMNS = [
    "dq_invalid_admission_time",
    "dq_invalid_ed_time",
    "dq_death_before_admission",
    "dq_death_after_discharge",
    "dq_expire_deathtime_mismatch",
]

pre = admissions_silver.select(
    F.count("*").alias("silver_rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_ids"),
    F.sum(F.when(F.col("hadm_id").isNull(), 1).otherwise(0)).alias("null_hadm_ids"),
    F.sum(F.when((F.col("hospital_expire_flag") == 1) & F.col("deathtime").isNull(), 1).otherwise(0)).alias("expired_but_no_deathtime"),
    *[F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c) for c in FLAG_COLUMNS],
    *[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"null_{c}") for c in FLAG_COLUMNS],
).first()
display(spark.createDataFrame([pre.asDict()]))

bronze_rows = admissions_bronze.count()
assert pre["silver_rows"] == bronze_rows, "Row count differs from Bronze"
assert pre["null_hadm_ids"] == 0, "NULL hadm_id present"
assert pre["unique_hadm_ids"] == pre["silver_rows"], "hadm_id is not unique"
for c in FLAG_COLUMNS:
    assert pre[f"null_{c}"] == 0, f"{c} contains NULLs"

if ENFORCE_PROFILED_COUNTS:
    for c, n in PROFILED_FLAG_COUNTS.items():
        assert pre[c] == n, f"{c}: expected {n}, got {pre[c]}"
    assert pre["expired_but_no_deathtime"] == PROFILED_EXPIRED_NO_DEATHTIME
    # dq_expire_deathtime_mismatch also covers hospital_expire_flag = 0 with a deathtime. A re-derivation from
    # the local source CSV (546,028 rows) found 11 expired-without-deathtime rows and 0 survivors with a
    # deathtime, so the flag total on that release is 11 (see docs/gold_governance_checks.md, Source readiness).
    assert pre["dq_expire_deathtime_mismatch"] == PROFILED_EXPIRED_NO_DEATHTIME

In [0]:
# publish (CREATE OR REPLACE; downstream Gold depends on this table)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
admissions_silver.writeTo(TARGET).using("delta").createOrReplace()

print(f"Published: {TARGET}")

In [0]:
# post-publish validation: ordered schema (pass-through columns keep their Bronze types)
SOURCE_COLUMNS = [
    "subject_id", "hadm_id", "admittime", "dischtime", "deathtime",
    "admission_type", "admit_provider_id", "admission_location",
    "discharge_location", "insurance", "language", "marital_status",
    "race", "edregtime", "edouttime", "hospital_expire_flag",
]
bronze_types = {f.name: f.dataType.simpleString() for f in admissions_bronze.schema.fields}
EXPECTED_SCHEMA = {c: bronze_types[c] for c in SOURCE_COLUMNS}
EXPECTED_SCHEMA.update({c: "boolean" for c in FLAG_COLUMNS})
EXPECTED_SCHEMA["_silver_processed_at"] = "timestamp"

published = spark.table(TARGET)
actual_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}
assert list(actual_schema.items()) == list(EXPECTED_SCHEMA.items()), f"Schema mismatch: {actual_schema}"

post = published.select(
    F.count("*").alias("rows"),
    F.countDistinct("hadm_id").alias("unique_hadm_ids"),
    F.sum(F.when(F.col("hadm_id").isNull(), 1).otherwise(0)).alias("null_hadm_ids"),
    *[F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c) for c in FLAG_COLUMNS],
    *[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"null_{c}") for c in FLAG_COLUMNS],
).first()

assert post["rows"] == bronze_rows, "Published row count differs from Bronze"
assert post["null_hadm_ids"] == 0
assert post["unique_hadm_ids"] == post["rows"]
for c in FLAG_COLUMNS:
    assert post[f"null_{c}"] == 0, f"{c} contains NULLs"
    assert post[c] == pre[c], f"{c} changed after publish"
if ENFORCE_PROFILED_COUNTS:
    for c, n in PROFILED_FLAG_COUNTS.items():
        assert post[c] == n, f"{c}: expected {n}, got {post[c]}"
print("Validated:", post.asDict())

Team note (not implemented): "null marital status to unknown". `marital_status` is nullable
in the documented schema (no NOT NULL) and the documentation does not define what NULL means,
so Silver preserves it as a source NULL and leaves `marital_status` unchanged; any default label belongs in a
downstream/Gold decision (project rule: do not fill NULLs). Source-file evidence for release data held
locally: 13,619 of 546,028 admissions (2.5%) have an empty `marital_status`. The cell below only inspects those rows.

In [0]:
display(spark.table(f"{BRONZE}.admissions").filter(F.col("marital_status").isNull()).limit(100))

In [0]:
dbutils.notebook.exit("admissions:OK")